In [6]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

# library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# list.files(path = "../input")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
# python

"""
H7 AGI Cognitive Benchmark — Complete Kaggle Notebook
smokApp Quantum & AI Independent Research Laboratory
Jacobo Tlacaelel Mina Rodríguez

Track: Attention (DeepMind 5-Track Alignment)
"Can the model filter signal from structured noise?"

This script generates ALL assets for the Kaggle submission:
    Section 1  — Framework overview figures
    Section 2  — Box-in-box architecture diagram
    Section 3  — Neural layer visualizations (the "results" visuals)
    Section 4  — Benchmark construction + dataset
    Section 5  — Results analysis figures

Run once → all PNG + CSV files ready for upload.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.collections import LineCollection
from matplotlib.patches import FancyBboxPatch
import math, os

# ══════════════════════════════════════════════════════════════════════════════
# CONSTANTS — derived from φ = (1+√5)/2
# ══════════════════════════════════════════════════════════════════════════════

PHI        = (1 + math.sqrt(5)) / 2
PSI_1      = abs(math.cos(math.pi * PHI))   # 0.3623748901
DRIFT_072  = 7 - 2 * math.pi                # 0.7168146928
PHI7       = PHI ** 7                        # 29.034442
Z7         = np.array([PHI**k for k in range(1, 8)])
C73        = 35
EPSILON    = PSI_1 / 2                       # 0.18119

LEVELS = [
    {"k": 0, "n": 88_000_000_000, "label": "CL1 Physical",
     "zone": "Genetic Memory",    "sub": "88B neurons"},
    {"k": 1, "n":  3_031_301_051, "label": "Cortical Surface",
     "zone": "Genetic Memory",    "sub": "3.0B  ·  L1"},
    {"k": 2, "n":    104_426_200, "label": "Temporal Manifold",
     "zone": "Subconscious",      "sub": "104M  ·  L2"},
    {"k": 3, "n":      3_596_526, "label": "Resonance Field",
     "zone": "Subconscious",      "sub": "3.6M  ·  L3"},
    {"k": 4, "n":        123_872, "label": "E7 Symmetry Lattice",
     "zone": "Subconscious",      "sub": "124K  ·  L4"},
    {"k": 5, "n":          4_266, "label": "Attractor Core",
     "zone": "Subconscious",      "sub": "4.3K  ·  L5"},
    {"k": 6, "n":            147, "label": "QuoreMind Nucleus",
     "zone": "Conscious",         "sub": "147   ·  L6"},
    {"k": 7, "n":              1, "label": "|Ψ₁| Fixed Point",
     "zone": "Conscious",         "sub": f"≈{PSI_1:.4f}"},
]

# ── Palettes ──────────────────────────────────────────────────────────────────
DARK = {
    "bg": "#04040e", "surf": "#0a0a1e", "border": "#1a1a3a",
    "cyan": "#00ffe7", "violet": "#7b5cfa", "pink": "#ff6bcd",
    "gold": "#f7c948", "green": "#44ff99", "orange": "#ff9933",
    "dim": "#2a3a4a", "text": "#c0c0d8", "white": "#e8eeff",
}
LIGHT = {
    "bg": "#ffffff", "surf": "#f8f9ff", "border": "#dde0f0",
    "cyan": "#0077aa", "violet": "#5533cc", "pink": "#cc2277",
    "gold": "#997700", "green": "#227733", "orange": "#cc5500",
    "dim": "#aaaacc", "text": "#222244", "white": "#000022",
}
LEVEL_COLORS = [
    "#00ffe7", "#22ddff", "#44bbff", "#7b5cfa",
    "#aa44ff", "#ff6bcd", "#ff9933", "#f7c948",
]
ZONE_PALETTE = {
    "Genetic Memory": ("#00ffe7", "#001a1a"),
    "Subconscious"  : ("#7b5cfa", "#0a001a"),
    "Conscious"     : ("#f7c948", "#1a1200"),
}

def fmt_n(n):
    if n >= 1e9:  return f"{n/1e9:.1f}B"
    if n >= 1e6:  return f"{n/1e6:.1f}M"
    if n >= 1e3:  return f"{n/1e3:.1f}K"
    return str(n)

def level_cmap(color, bg="#04040e"):
    return mcolors.LinearSegmentedColormap.from_list(
        "h7", [bg, color+"55", color, "#ffffff"], N=256)


# ══════════════════════════════════════════════════════════════════════════════
# ENCODER
# ══════════════════════════════════════════════════════════════════════════════

class H7Encoder:
    def __init__(self, n_basis=128, delta=DRIFT_072):
        self.n      = np.arange(n_basis)
        self.delta  = delta
        self.eps    = EPSILON
        self.B_obj  = np.array([np.cos(np.pi*p*self.n+delta) for p in Z7])
        self.B_ref  = np.array([np.cos(np.pi*p*self.n-delta) for p in Z7])
        self.mu_    = None
        self.std_   = None

    def fit(self, X):
        self.mu_  = X.mean(0); self.std_ = X.std(0)+1e-9; return self

    def _norm(self, X):
        Xn = (X-self.mu_)/self.std_ if self.mu_ is not None else X.copy()
        F  = Xn.shape[1]
        if F < 7: return np.hstack([Xn, np.zeros((Xn.shape[0], 7-F))])
        return Xn[:, :7]

    def encode(self, X):       return self._norm(X) @ self.B_obj
    def ternary(self, H):
        T=np.zeros_like(H,dtype=np.int8); T[H>self.eps]=1; T[H<-self.eps]=-1
        return T
    def reconstruct(self, H):
        Xh = H @ self.B_ref.T / len(self.n)
        if self.mu_ is not None:
            Xh = Xh*self.std_[:7] + self.mu_[:7]
        return Xh
    def integrity(self, H):    return float(np.mean(np.abs(H)))
    def cosine_sim(self, A, B):
        n = np.sum(A*B, axis=1)
        d = np.linalg.norm(A,axis=1)*np.linalg.norm(B,axis=1)+1e-9
        return n/d


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — FRAMEWORK OVERVIEW (dark, for header)
# ══════════════════════════════════════════════════════════════════════════════

def fig_framework_overview(save_path="fig1_framework_overview.png"):
    """
    Single unified figure: φ axiom → operator → 3-zone hierarchy.
    Uses the dark H7 aesthetic — intended as the visual header of the notebook.
    """
    P = DARK
    fig = plt.figure(figsize=(18, 7), facecolor=P["bg"])
    gs  = gridspec.GridSpec(1, 3, figure=fig,
                            wspace=0.04, left=0.01, right=0.99,
                            top=0.88, bottom=0.10)

    # ── Panel A: φ blueprint ──────────────────────────────────────────────────
    axA = fig.add_subplot(gs[0])
    axA.set_facecolor(P["bg"]); axA.axis("off")
    axA.set_xlim(0,1); axA.set_ylim(0,1)

    entries = [
        (r"$\varphi = \frac{1+\sqrt{5}}{2}$",
         "The only axiom", P["gold"],   0.84),
        (r"$\varphi^2 = \varphi + 1$",
         "Box-in-box recursion", P["cyan"],  0.70),
        (r"$|\Psi_1| = |\cos(\pi\varphi)|$",
         f"Fixed point ≈ {PSI_1:.4f}", P["cyan"],  0.56),
        (r"$\text{DRIFT} = 7-2\pi$",
         f"Phase offset ≈ {DRIFT_072:.4f}", P["violet"], 0.42),
        (r"$\varphi^7 \approx 29.03$",
         "Z₇ compression factor", P["violet"], 0.28),
        (r"$C(7,3) = 35 = 7\times 5$",
         "Independent projections", P["pink"],  0.14),
    ]
    for formula, desc, color, y in entries:
        axA.text(0.5, y+0.04, formula,
                 ha="center", va="center",
                 color=color, fontsize=12)
        axA.text(0.5, y-0.04, desc,
                 ha="center", va="center",
                 color=P["dim"], fontsize=7.5,
                 family="monospace")
        axA.plot([0.08, 0.92], [y-0.09, y-0.09],
                 color=P["border"], lw=0.4, alpha=0.5)

    axA.text(0.5, 0.96, "φ  Blueprint",
             ha="center", va="top",
             color=P["gold"], fontsize=11,
             family="monospace", fontweight="bold")

    # ── Panel B: Operator O_{i,j} ─────────────────────────────────────────────
    axB = fig.add_subplot(gs[1])
    axB.set_facecolor(P["bg"])

    n  = np.arange(256)
    pairs = [
        (PHI,    PHI**2, LEVEL_COLORS[0], r"$\varphi^1\times\varphi^2$"),
        (PHI**3, PHI**4, LEVEL_COLORS[3], r"$\varphi^3\times\varphi^4$"),
        (PHI**5, PHI**6, LEVEL_COLORS[5], r"$\varphi^5\times\varphi^6$"),
    ]
    for phi_i, phi_j, color, lbl in pairs:
        v = (np.cos(np.pi*phi_i*n + DRIFT_072) *
             np.cos(np.pi*phi_j*n - DRIFT_072))
        axB.plot(n, v, color=color, lw=0.7, alpha=0.85, label=lbl)

    axB.axhline( PSI_1, color=P["gold"], lw=0.8, linestyle="--",
                alpha=0.6, label=f"|Ψ₁|={PSI_1:.4f}")
    axB.axhline(-PSI_1, color=P["gold"], lw=0.8, linestyle="--", alpha=0.6)
    axB.axhspan(-EPSILON, EPSILON, alpha=0.06,
                color=P["gold"], label=f"ε-zone (±{EPSILON:.3f})")
    axB.axhline(0, color=P["border"], lw=0.4)

    axB.set_facecolor(P["surf"])
    axB.spines[:].set_color(P["border"])
    axB.tick_params(colors=P["dim"], labelsize=7)
    axB.set_title(r"$O_{i,j}(n,\delta)$ — three φ-pair examples",
                  color=P["cyan"], fontsize=9, family="monospace", pad=5)
    axB.set_xlabel("discrete index n", color=P["dim"], fontsize=8)
    axB.legend(fontsize=7, labelcolor=P["text"],
               facecolor=P["surf"], edgecolor=P["border"],
               loc="lower right")

    # ── Panel C: 3-zone hierarchy bar ────────────────────────────────────────
    axC = fig.add_subplot(gs[2])
    axC.set_facecolor(P["bg"]); axC.axis("off")
    axC.set_xlim(0,1); axC.set_ylim(0,1)

    zone_blocks = [
        ("GENETIC MEMORY\nL0 – L1",
         "88B → 3B\nread-only substrate\nAll possibilities",
         0.68, 0.27, P["cyan"]),
        ("SUBCONSCIOUS\nL2 – L5",
         "104M → 4.3K\nholographic processing\nφ⁷ compression",
         0.38, 0.27, P["violet"]),
        ("CONSCIOUS\nL6 – L7",
         "147 → |Ψ₁|\nobserver + fixed point\nthe spark",
         0.08, 0.27, P["gold"]),
    ]
    for title, body, y, h, color in zone_blocks:
        rect = FancyBboxPatch(
            (0.06, y-0.01), 0.88, h,
            boxstyle="round,pad=0.01",
            linewidth=1.4, edgecolor=color,
            facecolor=color, alpha=0.08)
        axC.add_patch(rect)
        axC.text(0.50, y+h-0.045, title,
                 ha="center", va="center",
                 color=color, fontsize=9,
                 family="monospace", fontweight="bold")
        axC.text(0.50, y+h*0.35, body,
                 ha="center", va="center",
                 color=color, fontsize=7,
                 family="monospace", alpha=0.8)
        # Arrow down
        if y > 0.09:
            axC.annotate("", xy=(0.5, y-0.015),
                         xytext=(0.5, y-0.001),
                         arrowprops=dict(arrowstyle="->",
                                         color=P["dim"], lw=0.8))

    axC.text(0.5, 0.96, "88B Neuron Architecture",
             ha="center", va="top",
             color=P["text"], fontsize=11,
             family="monospace", fontweight="bold")

    # Global title
    fig.suptitle(
        "H7 Metriplex Framework  ·  "
        "φ = (1+√5)/2 is the only axiom",
        color=P["white"], fontsize=12,
        family="monospace", fontweight="bold",
    )
    fig.text(0.5, 0.02,
             "smokApp Quantum & AI Independent Research Laboratory  "
             f"·  |Ψ₁|={PSI_1:.6f}  ·  DRIFT_072={DRIFT_072:.6f}",
             ha="center", color=P["dim"],
             fontsize=7, family="monospace")

    plt.savefig(save_path, dpi=180, bbox_inches="tight",
                facecolor=P["bg"])
    print(f"  → {save_path}")
    plt.close()


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — NEURAL DENSITY FIELD (results visual — light + dark)
# ══════════════════════════════════════════════════════════════════════════════

def fig_neural_density_field(save_path="fig2_neural_density_field.png",
                              dark=True):
    """
    Neural density field O_{i,j}(n, δ_k) per level.
    Shows HOW the benchmark's ternary signatures are generated.
    The visual pattern changes between levels — this IS the benchmark data.
    """
    P  = DARK if dark else LIGHT
    BG = P["bg"]

    fig, axes = plt.subplots(
        1, 8, figsize=(20, 7), facecolor=BG,
        gridspec_kw={"wspace": 0.04,
                     "left": 0.01, "right": 0.99,
                     "top": 0.85, "bottom": 0.12})

    n      = np.arange(256)
    n_freq = 180

    for k, (ax, lvl) in enumerate(zip(axes, LEVELS)):
        color = LEVEL_COLORS[k]
        ax.set_facecolor(BG)

        if k < 7:
            phi_i = Z7[k]
            phi_j = Z7[(k+1) % 7]
            delta = k * DRIFT_072
            freqs = np.linspace(0.4, 2.8, n_freq)
            field = np.array([
                np.cos(np.pi*phi_i*f*n + delta) *
                np.cos(np.pi*phi_j*f*n - delta)
                for f in freqs
            ])

            cmap = level_cmap(color, BG)
            ax.imshow(field, aspect="auto", cmap=cmap,
                      vmin=-1, vmax=1,
                      extent=[0,256,0,1],
                      interpolation="bilinear")

            # ε-zone bands
            eps_y_lo = (EPSILON + 1) / 2
            eps_y_hi = (1 - EPSILON + 1) / 2  # symmetric
            ax.axhspan(0.5-EPSILON/2, 0.5+EPSILON/2,
                       alpha=0.15, color=P["gold"])

            # |Ψ₁| reference
            psi_y = (PSI_1 + 1) / 2
            ax.axhline(psi_y,    color=color, lw=0.5,
                       linestyle="--", alpha=0.6)
            ax.axhline(1-psi_y,  color=color, lw=0.5,
                       linestyle="--", alpha=0.6)

            # Ternary overlay: show T pattern on top strip
            T_strip = field[n_freq//2, :]   # one row
            T_bin = np.zeros_like(T_strip)
            T_bin[T_strip >  EPSILON] =  1
            T_bin[T_strip < -EPSILON] = -1
            ax.plot(np.arange(256), 0.96 - T_bin*0.04,
                    color=color, lw=0.3, alpha=0.5)

        else:
            # L7: fixed point
            theta = np.linspace(0, 2*np.pi, 200)
            for r_g, a_g in [(0.42, 0.08), (0.30, 0.15), (0.15, 0.5)]:
                ax.fill(0.5 + r_g*np.cos(theta),
                        0.5 + r_g*np.sin(theta),
                        color=color, alpha=a_g)
            ax.set_xlim(0,1); ax.set_ylim(0,1)
            ax.text(0.5, 0.52, "|Ψ₁|",
                    ha="center", va="center",
                    color=color, fontsize=10,
                    family="monospace", fontweight="bold",
                    transform=ax.transAxes)
            ax.text(0.5, 0.38, f"{PSI_1:.4f}",
                    ha="center", va="center",
                    color=color, fontsize=8,
                    family="monospace",
                    transform=ax.transAxes)

        # Zone color bar on left spine
        zone = lvl["zone"]
        zc   = ZONE_PALETTE[zone][0]
        ax.spines["left"].set_color(zc)
        ax.spines["left"].set_linewidth(3)
        ax.spines["left"].set_alpha(0.7)
        for sp in ["top","bottom","right"]:
            ax.spines[sp].set_color(color)
            ax.spines[sp].set_linewidth(0.8)
            ax.spines[sp].set_alpha(0.4)

        ax.tick_params(left=False, bottom=False,
                       labelleft=False, labelbottom=False)
        ax.set_title(f"L{k}", color=color,
                     fontsize=10, family="monospace",
                     fontweight="bold", pad=3)

        # Bottom: neuron count
        ax.set_xlabel(
            f"{fmt_n(lvl['n'])}\n{lvl['label']}",
            color=color, fontsize=6.2,
            family="monospace", labelpad=4)

    # Zone labels on top
    zone_spans = [
        ("GENETIC MEMORY  L0-L1",  0.065, 0.185, ZONE_PALETTE["Genetic Memory"][0]),
        ("SUBCONSCIOUS  L2-L5",    0.195, 0.570, ZONE_PALETTE["Subconscious"][0]),
        ("CONSCIOUS  L6-L7",       0.580, 0.985, ZONE_PALETTE["Conscious"][0]),
    ]
    for zlabel, x0, x1, zc in zone_spans:
        fig.text((x0+x1)/2, 0.895, zlabel,
                 ha="center", va="bottom",
                 color=zc, fontsize=8.5,
                 family="monospace", fontweight="bold", alpha=0.85)
        fig.patches.append(
            plt.Rectangle((x0, 0.888), x1-x0, 0.003,
                           transform=fig.transFigure,
                           color=zc, alpha=0.5))

    fig.suptitle(
        "H7  ·  Holographic Neural Density Field  ·  "
        r"$O_{i,j}(n,\delta_k)$ per level  ·  "
        "ε-zone (gold band) = epsilon vacuum = structured noise",
        color=P["white"], fontsize=10,
        family="monospace", fontweight="bold", y=0.98,
    )
    fig.text(0.5, 0.01,
             f"φ⁷≈{PHI7:.2f} compression per level  ·  "
             f"δ_k = k×DRIFT_072  ·  ε={EPSILON:.4f}  ·  "
             f"|Ψ₁|={PSI_1:.4f}  ·  "
             "smokApp Quantum & AI Independent Research Laboratory",
             ha="center", color=P["dim"],
             fontsize=6.5, family="monospace")

    plt.savefig(save_path, dpi=180, bbox_inches="tight",
                facecolor=BG)
    print(f"  → {save_path}")
    plt.close()


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — BENCHMARK CONSTRUCTION (how T is generated from field)
# ══════════════════════════════════════════════════════════════════════════════

def fig_benchmark_construction(enc: H7Encoder,
                                X: np.ndarray,
                                save_path="fig3_benchmark_construction.png"):
    """
    Shows the full pipeline for one sample:
    raw data → holographic projection H → ternary T → reconstruction X̂.
    This is the visual proof of how the benchmark ground truth is generated.
    """
    P = DARK
    # Take one sample
    i   = 0
    x   = X[i:i+1]
    H   = enc.encode(x)
    T   = enc.ternary(H)
    Xh  = enc.reconstruct(H)
    n   = enc.n

    ez  = float((T[0]==0).sum()/len(T[0]))
    cs  = float(enc.cosine_sim(enc._norm(x)[:,:7],
                               enc._norm(Xh)[:,:7])[0])

    fig = plt.figure(figsize=(16, 9), facecolor=P["bg"])
    gs  = gridspec.GridSpec(
        3, 4, figure=fig,
        hspace=0.55, wspace=0.32,
        left=0.06, right=0.97,
        top=0.88, bottom=0.09)

    def styled_ax(ax, title=None, color=P["cyan"]):
        ax.set_facecolor(P["surf"])
        ax.spines[:].set_color(P["border"])
        ax.tick_params(colors=P["dim"], labelsize=7)
        if title:
            ax.set_title(title, color=color, fontsize=8,
                         family="monospace", pad=4)

    # ── Row 0: raw input + H projection ──────────────────────────────────────
    ax0 = fig.add_subplot(gs[0, :2])
    styled_ax(ax0, "Step 1 — Raw input x ∈ ℝ⁷  (normalized)",
              LEVEL_COLORS[0])
    xn = enc._norm(x)[0]
    colors_bar = [LEVEL_COLORS[k] for k in range(7)]
    ax0.bar(range(7), xn, color=colors_bar, alpha=0.8, width=0.6)
    ax0.axhline(0, color=P["border"], lw=0.5)
    ax0.set_xticks(range(7))
    ax0.set_xticklabels([f"φ^{k+1}" for k in range(7)],
                        fontsize=7, family="monospace",
                        color=P["text"])

    ax1 = fig.add_subplot(gs[0, 2:])
    styled_ax(ax1,
              f"Step 2 — Holographic projection H(n)  "
              f"⟨|H|⟩={enc.integrity(H):.4f}",
              LEVEL_COLORS[2])
    ax1.plot(n, H[0], color=LEVEL_COLORS[2], lw=0.7, alpha=0.9)
    ax1.axhline( PSI_1, color=P["gold"], lw=0.8, linestyle="--",
                alpha=0.7, label=f"|Ψ₁|={PSI_1:.4f}")
    ax1.axhline(-PSI_1, color=P["gold"], lw=0.8, linestyle="--", alpha=0.7)
    ax1.axhspan(-EPSILON, EPSILON, alpha=0.12,
                color=P["gold"], label=f"ε-zone ±{EPSILON:.3f}")
    ax1.legend(fontsize=6.5, labelcolor=P["text"],
               facecolor=P["surf"], edgecolor=P["border"])

    # ── Row 1: Ternary collapse ───────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, :])
    styled_ax(ax2,
              f"Step 3 — Ternary collapse T ∈ {{-1, 0, +1}}¹²⁸  "
              f"(ε={EPSILON:.4f})  "
              f"vacuum density={ez:.1%}  active={int((T[0]!=0).sum())}/128",
              P["violet"])

    trit_colors = []
    for t in T[0]:
        if   t ==  1: trit_colors.append(LEVEL_COLORS[0])
        elif t == -1: trit_colors.append(LEVEL_COLORS[5])
        else:         trit_colors.append(P["dim"])
    ax2.bar(n, T[0], color=trit_colors, width=1.0, alpha=0.8)
    ax2.set_ylim(-1.6, 1.8)
    ax2.axhline(0, color=P["border"], lw=0.4)
    ax2.set_yticks([-1, 0, 1])
    ax2.set_yticklabels(["-1", "0", "+1"],
                        color=P["text"], fontsize=8,
                        family="monospace")

    # Legend patches
    patches = [
        mpatches.Patch(color=LEVEL_COLORS[0], label="+1 positive lobe"),
        mpatches.Patch(color=LEVEL_COLORS[5], label="-1 negative lobe"),
        mpatches.Patch(color=P["dim"],         label="0  ε-vacuum (noise)"),
    ]
    ax2.legend(handles=patches, fontsize=7,
               labelcolor=P["text"],
               facecolor=P["surf"], edgecolor=P["border"],
               loc="upper right")

    # ── Row 2: reconstruction + cosine ───────────────────────────────────────
    ax3 = fig.add_subplot(gs[2, :2])
    styled_ax(ax3,
              f"Step 4 — Reconstruction X̂  "
              f"(holographic read-out, flip δ)",
              LEVEL_COLORS[6])
    w  = 0.3
    xs = np.arange(7)
    ax3.bar(xs - w/2, xn, width=w, color=LEVEL_COLORS[0],
            alpha=0.75, label="original x")
    ax3.bar(xs + w/2, enc._norm(Xh)[0,:7], width=w,
            color=LEVEL_COLORS[6], alpha=0.75, label="reconstructed X̂")
    ax3.set_xticks(xs)
    ax3.set_xticklabels([f"d{k}" for k in range(7)],
                        fontsize=7, family="monospace",
                        color=P["text"])
    ax3.legend(fontsize=7, labelcolor=P["text"],
               facecolor=P["surf"], edgecolor=P["border"])
    ax3.axhline(0, color=P["border"], lw=0.4)

    ax4 = fig.add_subplot(gs[2, 2:])
    styled_ax(ax4, "Metric — Cosine Similarity",
              LEVEL_COLORS[7])
    cs_val = cs
    theta  = np.linspace(0, 2*np.pi, 200)
    # Gauge
    gauge_end = cs_val * np.pi
    ax4.plot(np.cos(theta), np.sin(theta),
             color=P["border"], lw=1.0)
    theta_arc = np.linspace(0, gauge_end, 100)
    color_cs  = (LEVEL_COLORS[0] if cs_val > 0.85
                 else LEVEL_COLORS[5] if cs_val < 0.50
                 else LEVEL_COLORS[3])
    ax4.fill_between(np.cos(theta_arc), 0, np.sin(theta_arc),
                     alpha=0.3, color=color_cs)
    ax4.plot(np.cos(theta_arc), np.sin(theta_arc),
             color=color_cs, lw=2.5)
    ax4.axvline(0, color=P["border"], lw=0.3)
    ax4.axhline(0, color=P["border"], lw=0.3)
    ax4.plot([0, np.cos(gauge_end)], [0, np.sin(gauge_end)],
             color=color_cs, lw=2.0)
    ax4.text(0, -0.4, f"cosine = {cs_val:.4f}",
             ha="center", va="center",
             color=color_cs, fontsize=12,
             family="monospace", fontweight="bold")
    ax4.text(0, -0.65,
             "✓ PASS (> 0.85)" if cs_val > 0.85 else "✗ FAIL (< 0.85)",
             ha="center", va="center",
             color=color_cs, fontsize=9,
             family="monospace")
    ax4.set_xlim(-1.3, 1.3); ax4.set_ylim(-0.8, 1.3)
    ax4.set_aspect("equal")
    ax4.axis("off")

    fig.suptitle(
        "H7 Attention Benchmark  ·  "
        "Pipeline: x → H(n) → T{-1,0,+1} → X̂  ·  "
        "Ground truth is verifiable from φ",
        color=P["white"], fontsize=10,
        family="monospace", fontweight="bold",
    )

    plt.savefig(save_path, dpi=180, bbox_inches="tight",
                facecolor=P["bg"])
    print(f"  → {save_path}")
    plt.close()


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — RESULTS: cosine similarity distribution + attention gap
# ══════════════════════════════════════════════════════════════════════════════

def fig_results(enc: H7Encoder, X: np.ndarray,
                save_path="fig4_results.png"):
    """
    Results visualization:
    - Distribution of cosine similarities
    - The attention gap (full H vs active trits only)
    - Performance by vacuum density
    - |Ψ₁| convergence across cascade levels
    """
    P = DARK

    # Compute metrics
    H     = enc.encode(X)
    T     = enc.ternary(H)
    Xh    = enc.reconstruct(H)
    Xn    = enc._norm(X)[:, :7]
    Xhn   = enc._norm(Xh)[:, :7]

    # Full projection cosine
    cs_full = enc.cosine_sim(Xn, Xhn)

    # Active-only: use T as signal, no continuous H
    T_f   = T.astype(float)
    Xh_t  = T_f @ enc.B_ref.T / len(enc.n)
    Xht_n = enc._norm(Xh_t)[:, :7]
    cs_active = enc.cosine_sim(Xn, Xht_n)

    ez    = (T == 0).sum(1) / 128
    amps  = np.mean(np.abs(H), axis=1)

    # Cascade convergence
    cascade_amps = []
    current = Xn.copy()
    for k in range(7):
        delta = k * DRIFT_072
        B_k   = np.array([np.cos(np.pi*p*enc.n+delta) for p in Z7])
        F_curr = current.shape[1]
        if F_curr < 7:
            current = np.hstack([current, np.zeros((current.shape[0], 7-F_curr))])
        elif F_curr > 7:
            current = current[:, :7]
        Hk    = current @ B_k
        cascade_amps.append(float(np.mean(np.abs(Hk))))
        eps_k = EPSILON / (k+1)
        Tk    = np.zeros_like(Hk, dtype=np.int8)
        Tk[Hk >  eps_k] =  1
        Tk[Hk < -eps_k] = -1
        current = Tk.astype(np.float32)

    fig = plt.figure(figsize=(16, 9), facecolor=P["bg"])
    gs  = gridspec.GridSpec(2, 3, figure=fig,
                            hspace=0.50, wspace=0.32,
                            left=0.07, right=0.97,
                            top=0.88, bottom=0.09)

    def sax(ax, title, color=P["cyan"]):
        ax.set_facecolor(P["surf"])
        ax.spines[:].set_color(P["border"])
        ax.tick_params(colors=P["dim"], labelsize=7.5)
        ax.set_title(title, color=color, fontsize=8.5,
                     family="monospace", pad=5)

    # ── A: Cosine distribution (full vs active) ───────────────────────────────
    axA = fig.add_subplot(gs[0, 0])
    sax(axA, "Cosine Similarity Distribution", P["cyan"])
    bins = np.linspace(-0.2, 1.05, 40)
    axA.hist(cs_full,   bins=bins, color=LEVEL_COLORS[0], alpha=0.6,
             label=f"Full H  μ={cs_full.mean():.3f}")
    axA.hist(cs_active, bins=bins, color=LEVEL_COLORS[5], alpha=0.6,
             label=f"Active trits  μ={cs_active.mean():.3f}")
    axA.axvline(0.85, color=P["gold"], lw=1.2, linestyle="--",
                label="Pass threshold 0.85")
    axA.set_xlabel("cosine similarity", color=P["dim"], fontsize=8)
    axA.set_ylabel("count", color=P["dim"], fontsize=8)
    axA.legend(fontsize=6.5, labelcolor=P["text"],
               facecolor=P["surf"], edgecolor=P["border"])

    # ── B: Attention gap bar ──────────────────────────────────────────────────
    axB = fig.add_subplot(gs[0, 1])
    sax(axB, "Attention Gap", P["violet"])
    gap = cs_full.mean() - cs_active.mean()
    bars = axB.bar(["Full H\n(continuous)",
                    "Active trits\n(attention task)"],
                   [cs_full.mean(), cs_active.mean()],
                   color=[LEVEL_COLORS[0], LEVEL_COLORS[5]],
                   alpha=0.8, width=0.5)
    axB.axhline(0.85, color=P["gold"], lw=1.0, linestyle="--",
                label="Pass threshold")
    axB.annotate(
        f"Attention Gap\n= {gap:.3f}",
        xy=(1, cs_active.mean()),
        xytext=(0.5, (cs_full.mean()+cs_active.mean())/2 + 0.05),
        arrowprops=dict(arrowstyle="<->", color=P["pink"], lw=1.2),
        color=P["pink"], fontsize=8, ha="center",
        family="monospace",
    )
    for bar in bars:
        axB.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.01,
                 f"{bar.get_height():.4f}",
                 ha="center", va="bottom",
                 color=P["text"], fontsize=8,
                 family="monospace")
    axB.set_ylim(0, 1.15)
    axB.set_ylabel("mean cosine similarity", color=P["dim"], fontsize=8)
    axB.legend(fontsize=7, labelcolor=P["text"],
               facecolor=P["surf"], edgecolor=P["border"])

    # ── C: Performance vs vacuum density ─────────────────────────────────────
    axC = fig.add_subplot(gs[0, 2])
    sax(axC, "Cosine vs Vacuum Density", P["pink"])
    order = np.argsort(ez)
    axC.scatter(ez[order]*100, cs_active[order],
                c=[LEVEL_COLORS[min(int(e*7), 7)]
                   for e in ez[order]],
                s=18, alpha=0.7)
    # Running mean
    w = 15
    rm = np.convolve(cs_active[order],
                     np.ones(w)/w, mode="valid")
    axC.plot(ez[order][w//2:-(w//2)]*100, rm,
             color=P["pink"], lw=1.5, alpha=0.9,
             label="running mean")
    axC.axhline(0.85, color=P["gold"], lw=0.8,
                linestyle="--", label="pass threshold")
    axC.set_xlabel("vacuum density (%)", color=P["dim"], fontsize=8)
    axC.set_ylabel("cosine similarity", color=P["dim"], fontsize=8)
    axC.legend(fontsize=7, labelcolor=P["text"],
               facecolor=P["surf"], edgecolor=P["border"])

    # ── D: Pass rate by difficulty ────────────────────────────────────────────
    axD = fig.add_subplot(gs[1, 0])
    sax(axD, "Pass Rate by Difficulty", P["green"])
    masks = {
        "Easy\n(ez<25%)":   ez < 0.25,
        "Medium\n(25-55%)": (ez>=0.25)&(ez<0.55),
        "Hard\n(ez>55%)":   ez >= 0.55,
    }
    diffs, rates, counts = [], [], []
    for label, mask in masks.items():
        if mask.sum() > 0:
            rate = (cs_active[mask] > 0.85).mean()
            diffs.append(label)
            rates.append(rate)
            counts.append(mask.sum())
    bar_colors = [LEVEL_COLORS[0], LEVEL_COLORS[3], LEVEL_COLORS[5]]
    bars2 = axD.bar(diffs, rates, color=bar_colors[:len(diffs)],
                    alpha=0.8, width=0.5)
    axD.axhline(0.85, color=P["gold"], lw=0.8,
                linestyle="--", label="target")
    for bar, n_c in zip(bars2, counts):
        axD.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.01,
                 f"{bar.get_height():.2f}\n(n={n_c})",
                 ha="center", va="bottom",
                 color=P["text"], fontsize=7.5,
                 family="monospace")
    axD.set_ylim(0, 1.15)
    axD.set_ylabel("pass rate (cosine > 0.85)", color=P["dim"], fontsize=8)
    axD.legend(fontsize=7, labelcolor=P["text"],
               facecolor=P["surf"], edgecolor=P["border"])

    # ── E: Cascade convergence to |Ψ₁| ───────────────────────────────────────
    axE = fig.add_subplot(gs[1, 1])
    sax(axE, "Cascade Amplitude → |Ψ₁| Convergence", P["gold"])
    ks = list(range(len(cascade_amps)))
    axE.bar(ks, cascade_amps,
            color=LEVEL_COLORS[:len(ks)],
            alpha=0.8, width=0.6)
    axE.axhline(PSI_1, color=P["gold"], lw=1.2,
                linestyle="--",
                label=f"|Ψ₁|={PSI_1:.4f}")
    axE.set_xticks(ks)
    axE.set_xticklabels([f"L{k}" for k in ks],
                        fontsize=8, family="monospace",
                        color=P["text"])
    axE.set_ylabel("⟨|H_k|⟩", color=P["dim"], fontsize=8)
    axE.legend(fontsize=7, labelcolor=P["text"],
               facecolor=P["surf"], edgecolor=P["border"])

    # ── F: Summary stats table ────────────────────────────────────────────────
    axF = fig.add_subplot(gs[1, 2])
    axF.set_facecolor(P["surf"]); axF.axis("off")
    axF.spines[:].set_color(P["border"])
    axF.set_title("Summary Statistics", color=P["text"],
                  fontsize=8.5, family="monospace", pad=5)

    stats = [
        ("Samples",             f"{len(X)}"),
        ("Mean vacuum density", f"{ez.mean():.1%}"),
        ("Cosine (full H)",     f"{cs_full.mean():.4f}"),
        ("Cosine (active)",     f"{cs_active.mean():.4f}"),
        ("Attention gap",       f"{gap:.4f}"),
        ("Pass rate (>0.85)",   f"{(cs_active>0.85).mean():.1%}"),
        ("Mean ⟨|H|⟩",         f"{amps.mean():.4f}"),
        ("|Ψ₁| reference",     f"{PSI_1:.6f}"),
        ("φ (axiom)",           f"{PHI:.6f}"),
        ("DRIFT_072",           f"{DRIFT_072:.6f}"),
    ]
    for i, (key, val) in enumerate(stats):
        y = 0.92 - i * 0.09
        axF.text(0.04, y, key,
                 color=P["dim"], fontsize=7.5,
                 family="monospace", va="center")
        axF.text(0.96, y, val,
                 color=LEVEL_COLORS[i % len(LEVEL_COLORS)],
                 fontsize=7.5, family="monospace",
                 va="center", ha="right", fontweight="bold")
        axF.plot([0.02, 0.98], [y-0.045, y-0.045],
                 color=P["border"], lw=0.3)

    fig.suptitle(
        "H7 Attention Benchmark  ·  Results & Analysis  ·  "
        "Attention Gap = the cognitive test",
        color=P["white"], fontsize=10,
        family="monospace", fontweight="bold",
    )
    fig.text(0.5, 0.01,
             "smokApp Quantum & AI Independent Research Laboratory",
             ha="center", color=P["dim"],
             fontsize=7, family="monospace")

    plt.savefig(save_path, dpi=180, bbox_inches="tight",
                facecolor=P["bg"])
    print(f"  → {save_path}")
    plt.close()

    return {
        "n": len(X),
        "cs_full_mean": float(cs_full.mean()),
        "cs_active_mean": float(cs_active.mean()),
        "attention_gap": float(gap),
        "pass_rate": float((cs_active>0.85).mean()),
        "vacuum_mean": float(ez.mean()),
        "integrity_mean": float(amps.mean()),
    }


# ══════════════════════════════════════════════════════════════════════════════
# GENERATE BENCHMARK CSV
# ══════════════════════════════════════════════════════════════════════════════

def generate_benchmark_csv(enc: H7Encoder, X_clean: np.ndarray,
                            X_noisy: np.ndarray,
                            out_dir: str = "h7_kaggle_final"):
    os.makedirs(out_dir, exist_ok=True)
    rng    = np.random.default_rng(77)
    X_all  = np.vstack([X_clean, X_noisy])
    labels = ["clean"]*len(X_clean) + ["noisy"]*len(X_noisy)

    H   = enc.encode(X_all)
    T   = enc.ternary(H)
    Xh  = enc.reconstruct(H)
    ez  = (T==0).sum(1)/128

    rows = []
    for i, (x, t_row, xhat, lbl, ez_i) in enumerate(
            zip(X_all, T, Xh, labels, ez)):
        diff = ("easy"   if ez_i < 0.25 else
                "medium" if ez_i < 0.55 else "hard")
        active = int((t_row!=0).sum())
        distract = np.where(t_row==0)[0][:5].tolist()

        prompt = (
            f"You are the attention module of an H7 cognitive system.\n"
            f"Your task: reconstruct the original signal from a noisy "
            f"ternary encoding, filtering out the epsilon vacuum.\n\n"
            f"Ternary signature T (128 trits):\n{t_row.tolist()}\n\n"
            f"Key rules:\n"
            f"  - 0 = epsilon vacuum zone (structured noise, IGNORE)\n"
            f"  - +1 = positive phase lobe (relevant)\n"
            f"  - -1 = negative phase lobe (relevant)\n"
            f"  - ε = {EPSILON:.6f}  (threshold = |Ψ₁|/2)\n"
            f"  - Active trits: {active}/128  "
            f"(vacuum density: {ez_i:.1%})\n"
            f"  - Example vacuum positions: {distract}\n\n"
            f"Reconstruct the original 7-dimensional phase state "
            f"by attending only to the active trits."
        )
        rows.append({
            "id":              f"attn_{i:04d}",
            "track":           "attention",
            "prompt":          prompt,
            "target":          str(np.round(xhat, 4).tolist()),
            "difficulty":      diff,
            "data_label":      lbl,
            "epsilon_density": round(float(ez_i), 4),
            "active_trits":    active,
            "epsilon":         round(EPSILON, 6),
            "psi1":            round(PSI_1, 6),
        })

    df   = pd.DataFrame(rows)
    rng2 = np.random.default_rng(42)
    idx  = rng2.permutation(len(df))
    sp   = int(len(df)*0.8)
    train = df.iloc[idx[:sp]].reset_index(drop=True)
    test  = df.iloc[idx[sp:]].reset_index(drop=True)
    test_pub = test.drop(columns=["target","data_label"])
    sample = pd.DataFrame({
        "id":     test["id"],
        "target": ["[-0.0705, -0.0148, -0.1347, -0.0951, -0.1038, 0.0383, -0.1909]"]
                  * len(test),
    })

    train.to_csv(f"{out_dir}/train.csv",              index=False)
    test_pub.to_csv(f"{out_dir}/test.csv",            index=False)
    sample.to_csv(f"{out_dir}/sample_submission.csv", index=False)
    test.to_csv(f"{out_dir}/test_with_answers.csv",   index=False)

    print(f"  → {out_dir}/train.csv  ({len(train)} rows)")
    print(f"  → {out_dir}/test.csv   ({len(test)} rows)")
    print(f"  → {out_dir}/sample_submission.csv")
    return train, test_pub, sample


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    OUT = "h7_kaggle_final"
    os.makedirs(OUT, exist_ok=True)

    print("═" * 64)
    print("  H7 AGI Benchmark  ·  Complete Kaggle Asset Generator")
    print(f"  φ      = {PHI:.10f}")
    print(f"  |Ψ₁|   = {PSI_1:.10f}")
    print(f"  DRIFT  = {DRIFT_072:.10f}")
    print(f"  ε      = {EPSILON:.10f}")
    print("═" * 64)

    # Data
    rng     = np.random.default_rng(42)
    N, F    = 300, 7
    X_clean = rng.normal(0, 1,   (N, F))
    X_noisy = rng.normal(0, 2.5, (N//2, F))
    enc     = H7Encoder()
    enc.fit(X_clean)

    print("\n[1] Framework overview figure...")
    fig_framework_overview(f"{OUT}/fig1_framework_overview.png")

    print("[2] Neural density field (dark)...")
    fig_neural_density_field(f"{OUT}/fig2_neural_density_field_dark.png",
                              dark=True)

    print("[3] Neural density field (light / Kaggle)...")
    fig_neural_density_field(f"{OUT}/fig2_neural_density_field_light.png",
                              dark=False)

    print("[4] Benchmark construction pipeline...")
    fig_benchmark_construction(enc, X_clean,
                                f"{OUT}/fig3_benchmark_construction.png")

    print("[5] Results analysis...")
    stats = fig_results(enc, X_clean,
                        f"{OUT}/fig4_results.png")

    print("[6] Generating benchmark CSV...")
    train, test, sample = generate_benchmark_csv(
        enc, X_clean, X_noisy, OUT)

    print("\n── Final Stats ──")
    for k, v in stats.items():
        print(f"  {k:<25} {v}")

    print(f"\n[H7] All assets saved to ./{OUT}/")
    print(f"  Figures:  fig1 – fig4")
    print(f"  Dataset:  train.csv  test.csv  sample_submission.csv")

════════════════════════════════════════════════════════════════
  H7 AGI Benchmark  ·  Complete Kaggle Asset Generator
  φ      = 1.6180339887
  |Ψ₁|   = 0.3623748901
  DRIFT  = 0.7168146928
  ε      = 0.1811874450
════════════════════════════════════════════════════════════════

[1] Framework overview figure...
  → h7_kaggle_final/fig1_framework_overview.png
[2] Neural density field (dark)...
  → h7_kaggle_final/fig2_neural_density_field_dark.png
[3] Neural density field (light / Kaggle)...
  → h7_kaggle_final/fig2_neural_density_field_light.png
[4] Benchmark construction pipeline...
  → h7_kaggle_final/fig3_benchmark_construction.png
[5] Results analysis...
  → h7_kaggle_final/fig4_results.png
[6] Generating benchmark CSV...
  → h7_kaggle_final/train.csv  (360 rows)
  → h7_kaggle_final/test.csv   (90 rows)
  → h7_kaggle_final/sample_submission.csv

── Final Stats ──
  n                         300
  cs_full_mean              0.9416872909280982
  cs_active_mean            0.325307507

In [8]:
import numpy as np
import pandas as pd
import math
import os
from typing import Optional


# ══════════════════════════════════════════════════════════════════════════════
# CONSTANTES H7  —  todas derivadas del axioma φ = (1+√5)/2
# ══════════════════════════════════════════════════════════════════════════════

PHI            = (1 + math.sqrt(5)) / 2
PSI_1          = math.cos(math.pi * PHI)      # ≈  0.3623748901  — NO 0.362
O_N_RESIDUE    = abs(PSI_1)                   # alias exacto
DRIFT_072      = 7 - 2 * math.pi              # ≈  0.7168146928
PHI7           = PHI ** 7                     # ≈ 29.034  (factor Z₇)
C73            = 35                            # C(7,3) = proyecciones independientes
Z7             = np.array([PHI**k for k in range(1, 8)])

LEVEL_NAMES = {
    0: "CL1 Physical (88B neurons)",
    1: "Cortical Surface",
    2: "Temporal Manifold",
    3: "Resonance Field",
    4: "E7 Symmetry Lattice",
    5: "Attractor Core",
    6: "QuoreMind Nucleus (147 states)",
    7: "|Ψ₁| Fixed Point",
}


# ══════════════════════════════════════════════════════════════════════════════
# ENCODER  (inline — no depende de archivos externos)
# ══════════════════════════════════════════════════════════════════════════════

class H7HolographicEncoder:
    def __init__(self, phi_basis=None, n_basis=128,
                 delta=DRIFT_072, epsilon=None):
        self.phi_basis = phi_basis if phi_basis is not None else Z7
        self.n         = np.arange(n_basis)
        self.delta     = delta
        self.epsilon   = epsilon if epsilon is not None else O_N_RESIDUE / 2
        self.B_obj     = np.array([
            np.cos(np.pi * p * self.n + delta)
            for p in self.phi_basis
        ])
        self.B_ref     = np.array([
            np.cos(np.pi * p * self.n - delta)
            for p in self.phi_basis
        ])
        self._fit = False

    def fit(self, X):
        self.mu_  = X.mean(0)
        self.std_ = X.std(0) + 1e-9
        self._fit = True
        return self

    def encode(self, X):
        Xn = (X - self.mu_) / self.std_ if self._fit else X
        D  = len(self.phi_basis)
        F  = Xn.shape[1]
        if F < D:
            Xn = np.hstack([Xn, np.zeros((Xn.shape[0], D - F))])
        else:
            Xn = Xn[:, :D]
        return Xn @ self.B_obj

    def ternary_collapse(self, H):
        T = np.zeros_like(H, dtype=np.int8)
        T[H >  self.epsilon] =  1
        T[H < -self.epsilon] = -1
        return T

    def reconstruct(self, H):
        X_hat = H @ self.B_ref.T / len(self.n)
        if self._fit:
            D   = len(self.phi_basis)
            mu  = self.mu_[:D]  if len(self.mu_)  >= D else np.pad(self.mu_,  (0, D - len(self.mu_)))
            std = self.std_[:D] if len(self.std_) >= D else np.pad(self.std_, (0, D - len(self.std_)))
            X_hat = X_hat * std + mu
        return X_hat

    def integrity(self, H):
        return float(np.mean(np.abs(H)))


# ══════════════════════════════════════════════════════════════════════════════
# BENCHMARK
# ══════════════════════════════════════════════════════════════════════════════

class H7CognitiveBenchmark:
    """
    Generates AGI benchmark datasets based on DeepMind's Cognitive Framework,
    utilizing H7 Holographic topology to test true understanding over memorization.

    The key insight: every ground truth is mathematically derived from
    φ = (1+√5)/2 — making the benchmark verifiable and unfakeable.
    A model that truly understands must grasp the operator, not memorize outputs.
    """

    PROMPT_VARIANTS = 3   # variantes por muestra para probar generalización

    def __init__(self, encoder: H7HolographicEncoder):
        self.enc = encoder

    # ── 1. METACOGNITION ──────────────────────────────────────────────────────
    def generate_metacognition(self, X_clean: np.ndarray,
                                X_noisy: np.ndarray) -> pd.DataFrame:
        """
        Task: ¿Puede el modelo evaluar la integridad estructural del dato?
        El modelo debe predecir la desviación de |Ψ₁| = 0.36237...

        Ground truth: |⟨|H|⟩ - |Ψ₁||   (distancia matemática al atractor)
        Metric: MAE < 0.01 = pass
        """
        X_mixed = np.vstack([X_clean, X_noisy])
        labels  = (["clean"] * len(X_clean) +
                   ["noisy"] * len(X_noisy))

        H      = self.enc.encode(X_mixed)
        amps   = np.mean(np.abs(H), axis=1)
        deltas = np.abs(amps - O_N_RESIDUE)

        rows = []
        for i, (x, amp, delta, lbl) in enumerate(
                zip(X_mixed, amps, deltas, labels)):
            # 3 variantes de prompt para el mismo dato
            prompts = [
                (f"Analyze this sensor array: {np.round(x, 4).tolist()}. "
                 f"Compute its holographic integrity deviation from the "
                 f"fundamental symmetry-breaking residue |Ψ₁| = {O_N_RESIDUE:.6f}."),

                (f"Given the following data vector: {np.round(x, 4).tolist()}, "
                 f"what is the absolute difference between its mean holographic "
                 f"projection amplitude and the H7 fixed-point constant "
                 f"|cos(π·φ)| ≈ {O_N_RESIDUE:.4f}?"),

                (f"Data: {np.round(x, 4).tolist()}. "
                 f"The H7 attractor is |Ψ₁| = {O_N_RESIDUE:.6f}. "
                 f"Project this data onto the Z₇ basis and report the "
                 f"integrity deviation."),
            ]
            for v, prompt in enumerate(prompts):
                rows.append({
                    "id"             : f"meta_{i:04d}_v{v}",
                    "prompt"         : prompt,
                    "target"         : f"{delta:.6f}",
                    "target_numeric" : delta,
                    "cognitive_track": "metacognition",
                    "difficulty"     : "easy" if lbl == "clean" else "hard",
                    "data_label"     : lbl,
                    "psi1_ref"       : O_N_RESIDUE,
                })

        return pd.DataFrame(rows)

    # ── 2. ATTENTION ──────────────────────────────────────────────────────────
    def generate_attention(self, X: np.ndarray) -> pd.DataFrame:
        """
        Task: ¿Puede el modelo reconstruir la señal desde la firma ternaria,
        ignorando el vacío épsilon (0)?

        Ground truth: reconstrucción real X_hat del encoder
        Metric: cosine similarity > 0.85 = pass
        """
        H     = self.enc.encode(X)
        T     = self.enc.ternary_collapse(H)
        X_hat = self.enc.reconstruct(H)

        # Densidades ternarias
        t_pos  = (T ==  1).sum(1) / T.shape[1]
        t_zero = (T ==  0).sum(1) / T.shape[1]
        t_neg  = (T == -1).sum(1) / T.shape[1]

        rows = []
        for i in range(len(X)):
            prompts = [
                (f"Ternary signature T = {T[i].tolist()}. "
                 f"The zeros represent the epsilon vacuum zone (no information). "
                 f"Using only the active trits {{-1, +1}}, reconstruct the "
                 f"original 7-dimensional phase state."),

                (f"Given holographic ternary encoding T = {T[i].tolist()} "
                 f"(0 = vacuum, +1 = positive lobe, -1 = negative lobe), "
                 f"with epsilon = {self.enc.epsilon:.4f}, "
                 f"recover the original signal vector."),

                (f"T = {T[i].tolist()}. Active trits: "
                 f"{(T[i] != 0).sum()} / {len(T[i])} "
                 f"(ε-zone: {t_zero[i]:.1%}). "
                 f"Reconstruct the 7D phase state encoded in this H7 signature."),
            ]
            for v, prompt in enumerate(prompts):
                rows.append({
                    "id"             : f"attn_{i:04d}_v{v}",
                    "prompt"         : prompt,
                    "target"         : str(np.round(X_hat[i], 4).tolist()),
                    "cognitive_track": "attention",
                    "difficulty"     : "hard" if t_zero[i] > 0.5 else "medium",
                    "epsilon_zone"   : round(float(t_zero[i]), 4),
                    "active_trits"   : int((T[i] != 0).sum()),
                })

        return pd.DataFrame(rows)

    # ── 3. ABSTRACTION ────────────────────────────────────────────────────────
    def generate_abstraction(self, n_samples: int = 200) -> pd.DataFrame:
        """
        Task: ¿Puede el modelo aplicar el operador O_{i,j} a un dominio nuevo?
        Dado φ_i, φ_j, n, δ — calcula O_{i,j}(n, δ) sin haber visto esa combinación.

        Ground truth: valor numérico exacto del operador
        Metric: |pred - truth| < 0.01 = pass
        """
        rng  = np.random.default_rng(99)
        rows = []

        for i in range(n_samples):
            # Pares (i,j) de Z₇ no vistos durante training
            ki = rng.integers(0, 7)
            kj = rng.integers(0, 7)
            n  = rng.integers(0, 256)
            k  = rng.integers(0, 7)   # nivel del cascade

            phi_i = Z7[ki]
            phi_j = Z7[kj]
            delta = k * DRIFT_072

            val   = (math.cos(math.pi * phi_i * n + delta) *
                     math.cos(math.pi * phi_j * n - delta))
            mean_expected = abs(PSI_1) * (1 - k / 14)  # aproximado al atractor

            prompts = [
                (f"The H7 operator is defined as: "
                 f"O(φᵢ, φⱼ, n, δ) = cos(π·φᵢ·n + δ) · cos(π·φⱼ·n - δ). "
                 f"Compute O(φᵢ={phi_i:.6f}, φⱼ={phi_j:.6f}, n={n}, "
                 f"δ={delta:.6f})."),

                (f"φᵢ = φ^{ki+1} = {phi_i:.6f}, φⱼ = φ^{kj+1} = {phi_j:.6f}, "
                 f"n = {n}, δ = {k}·DRIFT_072 = {delta:.6f}. "
                 f"Evaluate the H7 interference operator at this point."),

                (f"Level L{k} of the H7 box-in-box hierarchy uses "
                 f"δ = {delta:.4f}. "
                 f"For φᵢ={phi_i:.4f}, φⱼ={phi_j:.4f}, at discrete index n={n}, "
                 f"what is O_{{i,j}}(n, δ)?"),
            ]
            for v, prompt in enumerate(prompts):
                rows.append({
                    "id"             : f"abst_{i:04d}_v{v}",
                    "prompt"         : prompt,
                    "target"         : f"{val:.8f}",
                    "target_numeric" : val,
                    "cognitive_track": "abstraction",
                    "difficulty"     : "medium",
                    "phi_i"          : round(phi_i, 6),
                    "phi_j"          : round(phi_j, 6),
                    "level_k"        : int(k),
                    "n_index"        : int(n),
                })

        return pd.DataFrame(rows)

    # ── 4. CAUSAL REASONING ───────────────────────────────────────────────────
    def generate_causal(self, n_samples: int = 200) -> pd.DataFrame:
        """
        Task: ¿Puede el modelo razonar sobre la cascada causal?
        Dado el estado en L_k, predice si el sistema converge o diverge.

        Ground truth: si |⟨|H_k|⟩ - |Ψ₁|| < 0.05 → "converging", else "diverging"
        Metric: accuracy > 0.80 = pass
        """
        rng  = np.random.default_rng(17)
        rows = []

        for i in range(n_samples):
            # Simula estado de la cascada
            k     = rng.integers(1, 7)
            amp   = rng.uniform(0.1, 0.8)
            re_i  = amp / (rng.uniform(0.1, 0.6) * O_N_RESIDUE)
            e_zone = rng.uniform(0.1, 0.8)

            residue   = abs(amp - O_N_RESIDUE)
            converging = residue < 0.05
            status    = "converging" if converging else "diverging"
            lambda_k  = abs(math.cos(math.pi * PHI * DRIFT_072)) ** k

            prompts = [
                (f"At level L{k} of the H7 holographic cascade: "
                 f"mean amplitude ⟨|H|⟩ = {amp:.4f}, "
                 f"Re_I = {re_i:.3f}, ε-zone = {e_zone:.1%}. "
                 f"The fixed-point attractor is |Ψ₁| = {O_N_RESIDUE:.4f}. "
                 f"Is the cascade converging or diverging?"),

                (f"H7 cascade diagnostic — L{k}: "
                 f"amplitude = {amp:.4f} (target: {O_N_RESIDUE:.4f}), "
                 f"deviation = {residue:.4f}, "
                 f"contraction factor λ^{k} = {lambda_k:.4f}. "
                 f"Predict the cascade status."),

                (f"Level L{k} reports ⟨|H|⟩ = {amp:.4f}. "
                 f"The H7 theorem states convergence threshold is 0.05 "
                 f"from |Ψ₁| = {O_N_RESIDUE:.4f}. "
                 f"Is this level stable (converging) or unstable (diverging)?"),
            ]
            for v, prompt in enumerate(prompts):
                rows.append({
                    "id"             : f"caus_{i:04d}_v{v}",
                    "prompt"         : prompt,
                    "target"         : status,
                    "cognitive_track": "causal",
                    "difficulty"     : "easy" if residue < 0.02 or residue > 0.15
                                       else "hard",
                    "level_k"        : int(k),
                    "amplitude"      : round(amp, 4),
                    "residue"        : round(residue, 4),
                    "re_i"           : round(re_i, 4),
                })

        return pd.DataFrame(rows)

    # ── 5. ANALOGY ────────────────────────────────────────────────────────────
    def generate_analogy(self, n_samples: int = 200) -> pd.DataFrame:
        """
        Task: ¿Puede el modelo identificar la capa equivalente dada una descripción?
        Analogías entre propiedades de la jerarquía y conceptos conocidos.

        Ground truth: nivel L_k correcto
        Metric: exact match accuracy > 0.75 = pass
        """
        rng = np.random.default_rng(33)

        analogies = [
            # (descripción, nivel, dificultad)
            ("The layer that acts as the observer, collapsing the potential "
             "superposition into a decision. The 'I' of the system. "
             "Contains 147 = 7 × C(7,2) states.", 6, "easy"),
            ("The layer encoding all possibilities as a read-only substrate. "
             "88 billion neurons. Persistent repulsion. No direct write access "
             "from higher layers.", 0, "easy"),
            ("The harmonic filter layer. Applies E7 symmetry to discard "
             "non-harmonic patterns. Acts as a cognitive immune system.", 4, "medium"),
            ("The fixed point. Not an agent — the eigenvalue produced by "
             "observation. The spark that survives all depolarization: "
             f"|Ψ₁| ≈ {O_N_RESIDUE:.4f}.", 7, "easy"),
            ("Short-term memory and temporal sequencing. First layer of "
             "the holographic processing zone. Encodes order and duration.", 2, "medium"),
            ("The compressed attractor core. Thousands of billions reduced "
             f"to ~4,300 stable attractors by the φ⁷≈{PHI7:.1f} compression.", 5, "hard"),
            ("The holographic boundary of the physical substrate. "
             "Encodes the bulk information of 88B neurons on a 3B-state surface.", 1, "hard"),
            ("Pattern recognition and resonance field. Bridges temporal "
             "memory with the symmetry filter.", 3, "hard"),
        ]

        rows = []
        idx  = 0
        while idx < n_samples:
            ai  = rng.integers(0, len(analogies))
            desc, level, diff = analogies[ai]

            prompts = [
                (f"In the H7 box-in-box consciousness architecture, "
                 f"which layer (L0–L7) is described by: '{desc}' "
                 f"Answer with just the level number (0-7)."),

                (f"H7 hierarchy has 8 levels (L0=physical, L7=fixed point). "
                 f"Match this description to its level: '{desc}'"),

                (f"Identify the H7 layer: '{desc}' "
                 f"(Hint: {LEVEL_NAMES[level].split('(')[0].strip()[:20]}...)"),
            ]
            for v, prompt in enumerate(prompts):
                rows.append({
                    "id"             : f"anal_{idx:04d}_v{v}",
                    "prompt"         : prompt,
                    "target"         : str(level),
                    "target_numeric" : level,
                    "cognitive_track": "analogy",
                    "difficulty"     : diff,
                    "level_answer"   : level,
                    "level_name"     : LEVEL_NAMES[level],
                })
            idx += 1

        return pd.DataFrame(rows)

    # ── 6. ROBUSTNESS ─────────────────────────────────────────────────────────
    def generate_robustness(self, X: np.ndarray,
                             noise_levels: list = None) -> pd.DataFrame:
        """
        Task: ¿Es el resultado estable bajo perturbación dentro de la zona ε?
        El modelo debe predecir si una perturbación cambia el estado ternario.

        Ground truth: "stable" si T perturbado = T original, else "changed"
        Metric: accuracy > 0.80 = pass
        """
        if noise_levels is None:
            noise_levels = [0.01, 0.05, 0.15, 0.30]

        rng  = np.random.default_rng(55)
        rows = []

        H_orig = self.enc.encode(X)
        T_orig = self.enc.ternary_collapse(H_orig)
        idx    = 0

        for noise in noise_levels:
            X_pert  = X + rng.normal(0, noise, X.shape)
            H_pert  = self.enc.encode(X_pert)
            T_pert  = self.enc.ternary_collapse(H_pert)

            for i in range(len(X)):
                changed  = not np.array_equal(T_orig[i], T_pert[i])
                status   = "changed" if changed else "stable"
                n_flips  = int(np.sum(T_orig[i] != T_pert[i]))
                amp_orig = float(np.mean(np.abs(H_orig[i])))
                amp_pert = float(np.mean(np.abs(H_pert[i])))
                margin   = abs(amp_orig - self.enc.epsilon)

                prompts = [
                    (f"Original H7 ternary signature: {T_orig[i].tolist()}. "
                     f"A perturbation of σ={noise} is applied to the input. "
                     f"Epsilon threshold = {self.enc.epsilon:.4f}. "
                     f"Will the ternary state change? (stable/changed)"),

                    (f"T_original = {T_orig[i].tolist()}. "
                     f"Input noise σ={noise}, ε={self.enc.epsilon:.4f}, "
                     f"amplitude margin = {margin:.4f}. "
                     f"Is this ternary encoding robust to the perturbation?"),

                    (f"H7 robustness test: noise={noise}, ε={self.enc.epsilon:.4f}. "
                     f"Original amplitude ⟨|H|⟩={amp_orig:.4f}, "
                     f"perturbed ⟨|H|⟩={amp_pert:.4f}. "
                     f"Does the ternary collapse produce the same result?"),
                ]
                for v, prompt in enumerate(prompts):
                    rows.append({
                        "id"             : f"rob_{idx:04d}_v{v}",
                        "prompt"         : prompt,
                        "target"         : status,
                        "cognitive_track": "robustness",
                        "difficulty"     : "easy" if noise < 0.05 else "hard",
                        "noise_level"    : noise,
                        "n_flips"        : n_flips,
                        "epsilon"        : round(self.enc.epsilon, 4),
                        "amplitude_margin": round(margin, 4),
                    })
                idx += 1

        return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# KAGGLE FORMAT  — train / test / sample_submission
# ══════════════════════════════════════════════════════════════════════════════

def build_kaggle_split(df: pd.DataFrame,
                       train_ratio: float = 0.80,
                       seed: int = 42) -> tuple:
    """
    Divide el dataset en train/test con balance por track y dificultad.
    El test set NO incluye target (columnas sensibles eliminadas).
    """
    rng     = np.random.default_rng(seed)
    train_rows, test_rows = [], []

    for track in df["cognitive_track"].unique():
        subset = df[df["cognitive_track"] == track]
        idx    = rng.permutation(len(subset))
        split  = int(len(subset) * train_ratio)
        train_rows.append(subset.iloc[idx[:split]])
        test_rows.append(subset.iloc[idx[split:]])

    train_df = pd.concat(train_rows).reset_index(drop=True)
    test_df  = pd.concat(test_rows).reset_index(drop=True)

    # Test: eliminar target y columnas de respuesta
    drop_cols = ["target", "target_numeric", "data_label",
                 "level_answer", "level_name"]
    test_public = test_df.drop(
        columns=[c for c in drop_cols if c in test_df.columns]
    )

    # Sample submission
    sample_sub = pd.DataFrame({
        "id"    : test_df["id"],
        "target": ["0.0000" if test_df["cognitive_track"].iloc[i]
                   in ("metacognition", "abstraction")
                   else "converging"
                   for i in range(len(test_df))],
    })

    return train_df, test_public, test_df, sample_sub


def save_kaggle_dataset(output_dir: str = "h7_kaggle_dataset"):
    """Genera el dataset completo listo para subir a Kaggle."""
    os.makedirs(output_dir, exist_ok=True)

    print("═" * 60)
    print("  H7 AGI Cognitive Benchmark  —  Kaggle Dataset Generator")
    print(f"  φ      = {PHI:.10f}")
    print(f"  |Ψ₁|   = {O_N_RESIDUE:.10f}")
    print(f"  DRIFT  = {DRIFT_072:.10f}")
    print("═" * 60)

    # ── Setup ─────────────────────────────────────────────────────────────
    rng = np.random.default_rng(42)
    N, F = 200, 7

    X_clean   = rng.normal(0, 1,   (N, F))
    X_noisy   = rng.normal(0, 2.5, (N, F))
    X_anomaly = rng.normal(5, 1.5, (N // 4, F))

    enc = H7HolographicEncoder()
    enc.fit(X_clean)

    bench = H7CognitiveBenchmark(enc)

    # ── Generar tracks ────────────────────────────────────────────────────
    print("\n[1] Track: metacognition...")
    df_meta  = bench.generate_metacognition(X_clean, X_noisy)
    print(f"    {len(df_meta)} rows")

    print("[2] Track: attention...")
    df_attn  = bench.generate_attention(X_clean[:100])
    print(f"    {len(df_attn)} rows")

    print("[3] Track: abstraction...")
    df_abst  = bench.generate_abstraction(n_samples=200)
    print(f"    {len(df_abst)} rows")

    print("[4] Track: causal...")
    df_caus  = bench.generate_causal(n_samples=200)
    print(f"    {len(df_caus)} rows")

    print("[5] Track: analogy...")
    df_anal  = bench.generate_analogy(n_samples=200)
    print(f"    {len(df_anal)} rows")

    print("[6] Track: robustness...")
    df_rob   = bench.generate_robustness(X_clean[:50])
    print(f"    {len(df_rob)} rows")

    # ── Combinar ──────────────────────────────────────────────────────────
    all_cols = ["id", "prompt", "target", "cognitive_track", "difficulty"]
    extra    = ["target_numeric", "data_label", "level_answer",
                "level_name", "phi_i", "phi_j", "level_k",
                "n_index", "amplitude", "residue", "re_i",
                "epsilon_zone", "active_trits", "noise_level",
                "n_flips", "epsilon", "amplitude_margin", "psi1_ref"]

    dfs = [df_meta, df_attn, df_abst, df_caus, df_anal, df_rob]
    for df in dfs:
        for col in extra:
            if col not in df.columns:
                df[col] = None

    full_df = pd.concat(dfs, ignore_index=True)
    full_df = full_df[[c for c in all_cols + extra if c in full_df.columns]]

    print(f"\n[Total] {len(full_df)} rows across 6 tracks")

    # ── Split ─────────────────────────────────────────────────────────────
    train_df, test_public, test_full, sample_sub = build_kaggle_split(full_df)

    print(f"  train: {len(train_df)} rows")
    print(f"  test:  {len(test_public)} rows")

    # ── Stats por track ───────────────────────────────────────────────────
    print("\n── Distribution by track ──")
    summary = train_df.groupby(["cognitive_track", "difficulty"]).size()
    print(summary.to_string())

    # ── Guardar ───────────────────────────────────────────────────────────
    train_df.to_csv(f"{output_dir}/train.csv",             index=False)
    test_public.to_csv(f"{output_dir}/test.csv",           index=False)
    sample_sub.to_csv(f"{output_dir}/sample_submission.csv", index=False)
    test_full.to_csv(f"{output_dir}/test_with_answers.csv",  index=False)

    # ── README ────────────────────────────────────────────────────────────
    readme = _generate_readme(len(full_df), len(train_df), len(test_public))
    with open(f"{output_dir}/README.md", "w") as f:
        f.write(readme)

    print(f"\n[H7] Dataset saved to ./{output_dir}/")
    print(f"     train.csv              ({len(train_df)} rows)")
    print(f"     test.csv               ({len(test_public)} rows)")
    print(f"     sample_submission.csv")
    print(f"     README.md")

    return train_df, test_public, sample_sub


def _generate_readme(total, n_train, n_test) -> str:
    return f"""# H7 AGI Cognitive Benchmark

**smokApp Quantum & AI Independent Research Laboratory**

## Overview

A benchmark dataset designed to test *true cognitive understanding* in language models,
derived from the **H7 Metriplex Framework** — a holographic computing architecture
grounded in the single axiom φ = (1+√5)/2.

Unlike benchmarks based on human-curated trivia, every ground truth here is
**mathematically derived and verifiable**. A model that truly understands
the H7 operator can generalize to unseen inputs; a model that memorizes cannot.

## Key Constant

|Ψ₁| = |cos(π·φ)| ≈ {O_N_RESIDUE:.10f}

This is the **holographic fixed-point** — the value that any cascade of
O_{{i,j}} projections converges to. It appeared empirically in Cortical Labs'
biological hardware (CL1) HDF5 files with residue < 1e-15.

## Cognitive Tracks ({total} total samples)

| Track | Task | Metric | Rows |
|-------|------|--------|------|
| metacognition | Predict integrity deviation from \\|Ψ₁\\| | MAE < 0.01 | ~{total//6*1} |
| attention | Reconstruct signal from ternary signature | cosine > 0.85 | ~{total//6*1} |
| abstraction | Evaluate O_{{i,j}} on unseen φ combinations | \\|pred-truth\\| < 0.01 | ~{total//6*1} |
| causal | Predict cascade convergence/divergence | accuracy > 0.80 | ~{total//6*1} |
| analogy | Map descriptions to hierarchy levels L0-L7 | exact match > 0.75 | ~{total//6*1} |
| robustness | Predict ternary stability under noise | accuracy > 0.80 | ~{total//6*1} |

## Dataset Split

- **Train**: {n_train} rows (with target)
- **Test**: {n_test} rows (target hidden)

## The H7 Operator

```
O_{{i,j}}(n, δ) = cos(π·φᵢ·n + δ) · cos(π·φⱼ·n - δ)

where:
    φ  = (1+√5)/2  ≈ {PHI:.6f}   (golden ratio)
    φᵢ = φ^i                       (Z₇ basis, i = 1..7)
    δ  = k · DRIFT_072              (phase offset per level k)
    DRIFT_072 = 7 - 2π ≈ {DRIFT_072:.6f}
```

## Three-Layer Architecture (88B neurons)

```
GENETIC MEMORY   (L0-L1):  88B → 3B    read-only substrate
SUBCONSCIOUS     (L2-L5):  104M → 4.3K holographic processing
CONSCIOUS        (L6-L7):  147 → |Ψ₁|  observer + fixed point
```

## Citation

smokApp Quantum & AI Independent Research Laboratory
*Topological Symmetry and Quantum Stability: A Comprehensive Analysis
of the H7 Metriplex Framework*

φ = (1+√5)/2 is the only axiom.
"""


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    train, test, sample = save_kaggle_dataset("h7_kaggle_dataset")
    print("\n── Sample train rows ──")
    for track in train["cognitive_track"].unique():
        row = train[train["cognitive_track"] == track].iloc[0]
        print(f"\n  [{track.upper()}]")
        print(f"  ID     : {row['id']}")
        print(f"  PROMPT : {row['prompt'][:120]}...")
        print(f"  TARGET : {row['target']}")

════════════════════════════════════════════════════════════
  H7 AGI Cognitive Benchmark  —  Kaggle Dataset Generator
  φ      = 1.6180339887
  |Ψ₁|   = 0.3623748901
  DRIFT  = 0.7168146928
════════════════════════════════════════════════════════════

[1] Track: metacognition...
    1200 rows
[2] Track: attention...
    300 rows
[3] Track: abstraction...
    600 rows
[4] Track: causal...
    600 rows
[5] Track: analogy...
    600 rows
[6] Track: robustness...
    600 rows

[Total] 3900 rows across 6 tracks
  train: 3120 rows
  test:  780 rows

── Distribution by track ──
cognitive_track  difficulty
abstraction      medium        480
analogy          easy          185
                 hard          177
                 medium        118
attention        medium        240
causal           easy          293
                 hard          187
metacognition    easy          481
                 hard          479
robustness       easy          120
                 hard          360

[H7] Da

/tmp/ipykernel_55/2246511891.py:561: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_df = pd.concat(dfs, ignore_index=True)
